In [1]:
!nvidia-smi

Thu Dec  5 16:24:51 2024       
+---------------------------------------------------------------------------------------+
| NVIDIA-SMI 535.104.05             Driver Version: 535.104.05   CUDA Version: 12.2     |
|-----------------------------------------+----------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id        Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |         Memory-Usage | GPU-Util  Compute M. |
|                                         |                      |               MIG M. |
|=========================================+======================+======================|
|   0  NVIDIA A100-SXM4-40GB          Off | 00000000:00:04.0 Off |                    0 |
| N/A   29C    P0              46W / 400W |      2MiB / 40960MiB |      0%      Default |
|                                         |                      |             Disabled |
+-----------------------------------------+----------------------+--

In [2]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"  # GPU ID를 0으로 설정
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'max_split_size_mb:64'

In [3]:
from google.colab import userdata
read_token = userdata.get('huggingface_read')

In [4]:
!huggingface-cli login --token $read_token

The token has not been saved to the git credentials helper. Pass `add_to_git_credential=True` in this function directly or `--add-to-git-credential` if using via `huggingface-cli` if you want to set the git credential as well.
Token is valid (permission: read).
The token `Read` has been saved to /root/.cache/huggingface/stored_tokens
Your token has been saved to /root/.cache/huggingface/token
Login successful.
The current active token is: `Read`


In [5]:
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel, PeftConfig

In [6]:
#연합학습 결과물인 어댑터 모델은 미리 허깅페이스에 업로드해놓아야 한다
model_1 = "prayslaks/python_code_instructions_18k_alpaca_20000_fedavg_c4s2_i10_b4a1_l512_r32a64_20241030071530"
model_2 = "prayslaks/alpaca-cleaned_20000_fedavg_c4s2_i10_b4a1_l512_r32a64_20241111092921"
model_3 = "prayslaks/flask_evaluation_2000_fedavg_c5s4_i10_b4a1_l2048_r16a32_20241120145940"
model_4 = "prayslaks/wikimedia_wikipedia_1M_1000000_fedavg_c4s2_i10_b4a1_l2048_r16a32_20241201054722"

adapter_id = model_4

In [7]:
#Peft 어댑터의 설정 획득
config = PeftConfig.from_pretrained(adapter_id, subfolder="checkpoint-10")
print(config.base_model_name_or_path)

/usr/local/lib/python3.10/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


checkpoint-10/adapter_config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

mistralai/Mistral-7B-Instruct-v0.3


In [8]:
model = AutoModelForCausalLM.from_pretrained(config.base_model_name_or_path)

config.json:   0%|          | 0.00/601 [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/23.9k [00:00<?, ?B/s]

model-00001-of-00003.safetensors:   0%|          | 0.00/4.95G [00:00<?, ?B/s]

model-00002-of-00003.safetensors:   0%|          | 0.00/5.00G [00:00<?, ?B/s]

model-00003-of-00003.safetensors:   0%|          | 0.00/4.55G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

In [9]:
tokenizer = AutoTokenizer.from_pretrained(adapter_id, subfolder="checkpoint-10")

checkpoint-10/tokenizer_config.json:   0%|          | 0.00/141k [00:00<?, ?B/s]

tokenizer.model:   0%|          | 0.00/587k [00:00<?, ?B/s]

checkpoint-10/special_tokens_map.json:   0%|          | 0.00/438 [00:00<?, ?B/s]

In [10]:
model = PeftModel.from_pretrained(model, adapter_id, subfolder='checkpoint-10')

adapter_model.safetensors:   0%|          | 0.00/27.3M [00:00<?, ?B/s]

In [11]:
import torch
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

PeftModelForCausalLM(
  (base_model): LoraModel(
    (model): MistralForCausalLM(
      (model): MistralModel(
        (embed_tokens): Embedding(32768, 4096)
        (layers): ModuleList(
          (0-31): 32 x MistralDecoderLayer(
            (self_attn): MistralSdpaAttention(
              (q_proj): lora.Linear(
                (base_layer): Linear(in_features=4096, out_features=4096, bias=False)
                (lora_dropout): ModuleDict(
                  (default): Dropout(p=0.05, inplace=False)
                )
                (lora_A): ModuleDict(
                  (default): Linear(in_features=4096, out_features=16, bias=False)
                )
                (lora_B): ModuleDict(
                  (default): Linear(in_features=16, out_features=4096, bias=False)
                )
                (lora_embedding_A): ParameterDict()
                (lora_embedding_B): ParameterDict()
                (lora_magnitude_vector): ModuleDict()
              )
              (k_proj): 

GPU를 이용하면 빠르면 1초부터 늦으면 10초 정도에 추론을 완료할 수 있다. 프로메테우스의 경우 VRAM을 27GB 정도 필요로 하며 코랩에서는 A100을 사용해야지만 GPU에 해당 모델을 로드할 수 있다.

In [14]:
# 토큰화 및 텐서 변환
input_text = ["Explain about Anarchism.",
              "Explain about Albedo",
              "Explain about ASCII art",
              "Explain about Apple II?",
              "Explain about Blade Runner",]


for text in input_text:
    inputs = tokenizer(text, return_tensors="pt").to(device)

    # 모델 추론
    with torch.no_grad():
        outputs = model.generate(inputs['input_ids'], max_length=200)

    # 결과 텍스트 변환
    generated_text = tokenizer.decode(outputs[0], skip_special_tokens=True)
    print(generated_text + '\n\n')

The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:None for open-end generation.
The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:None for open-end generation.


Explain about Anarchism.

Anarchism is a political philosophy that advocates self-governed societies, without a central government, and without any coercive forms of hierarchy or oppression. Anarchists seek to create a society in which individuals are free to make their own choices and to live their own lives, without interference from others.

Anarchism is often associated with the idea of "no government," but this is not entirely accurate. While anarchists do not believe in the need for a central government, they do not necessarily advocate for a complete absence of all forms of organization or structure. Instead, they believe that individuals should be free to form their own communities and organizations, as long as they do not infringe upon the rights of others.

Anarchism is a diverse philosophy, with many different schools of thought and approaches. Some anarchists are pacifists, while others advocate for the use of violence in certain circumstances.




The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:None for open-end generation.


Explain about Albedo and its importance in climate change

Albedo is a measure of the reflectivity of a surface. It is defined as the ratio of the reflected radiation to the incident radiation. Albedo is usually expressed as a decimal fraction or percentage. A surface with an albedo of 1.0 reflects all incident radiation, while a surface with an albedo of 0.0 absorbs all incident radiation.

Albedo is an important factor in the Earth's climate system because it affects the amount of solar radiation that is absorbed by the Earth's surface. Darker surfaces, such as forests and oceans, have lower albedos and absorb more solar radiation, while lighter surfaces, such as snow and ice, have higher albedos and reflect more solar radiation.

The albedo of the Earth's surface has changed over time due to changes in the distribution of land and water, vegetation




The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:None for open-end generation.


Explain about ASCII art

ASCII art is a graphic design technique that uses printable characters from the ASCII standard to create images. The term is also used to refer to images so created. ASCII art is a form of computer art, and is a precursor to bitmap graphics.

ASCII art is created by using the characters in the ASCII character set to represent pixels in an image. The characters are usually arranged in rows and columns, with each character representing a pixel in the image. The characters can be arranged in a variety of ways to create different effects, such as shading, textures, and patterns.

ASCII art can be created using a variety of tools, including text editors, programming languages, and specialized software. It can also be created by hand, using a keyboard and a text editor. ASCII art can be displayed on a variety of devices, including computers, smartphones, and other electronic




The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:None for open-end generation.


Explain about Apple II?

Apple II is a series of personal computers designed, manufactured, and sold by Apple Computer, Inc. (now Apple Inc.) from 1977 to 1993. The Apple II was the first successful personal computer, and it was the first to sell more than one million units. It was also the first to have a color graphics card, the first to have a built-in sound card, and the first to have a built-in floppy disk drive.

The Apple II was introduced in 1977 and was the first computer to be sold by Apple. It was designed by Steve Wozniak and Steve Jobs, and it was the first computer to use the Apple II operating system. The Apple II was a popular computer for home and business use, and it was used for a wide range of applications, including word processing, spreadsheets, databases, and games.



Explain about Blade Runner 2049.

Blade Runner 2049 is a 2017 American neo-noir science fiction film directed by Denis Villeneuve and written by Hampton Fancher and Michael Green. It is a sequel to